In [1]:
import pandas as pd
import numpy as np
import regex as re
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin
#!pip install pdfminer.six
from pdfminer.high_level import extract_text

nltk.download('stopwords')
nltk.download('punkt_tab')

df = pd.read_csv('job_title_des.csv')
stop_words = set(stopwords.words('english'))

def cleantext(text):
  '''
  standardizes case, removes stop words
  '''
  text = text.lower()
  text = re.sub(r'\W+', ' ', text)
  words = word_tokenize(text)
  words = [word for word in words if word not in stop_words]
  return ' '.join(words)

df['cleanDesc'] = df['Job Description'].apply(cleantext)
descForEmbed = df['cleanDesc'].tolist()

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(descForEmbed) ##embeds tokenized job descriptions in vector space
#print(df.head())

kmeans = KMeans(n_clusters=20, random_state=0, n_init="auto").fit(embeddings) ##cluster embeddings


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [52]:
###summarize by cluster -- counts and closest job title to center of each cluster
df['cluster'] = kmeans.fit_predict(embeddings).tolist()
summary = df.groupby('cluster', as_index=False)['Job Title'].agg('count')


centroids = kmeans.cluster_centers_
#print(centroids)
closest_points = pairwise_distances_argmin(embeddings, centroids) ###can the distance function of cosign similarity be slotted in here for more accuracy?
#print(closest_points)
title_lookup = dict(zip(closest_points, df['Job Title'][closest_points]))
print(title_lookup)
summary['Central Title'] = summary['cluster'].map(title_lookup)
print(summary)

{12: 'Machine Learning', 17: 'iOS Developer', 2: 'Machine Learning', 8: 'DevOps Engineer', 1: 'Django Developer', 4: 'Full Stack Developer', 10: 'Database Administrator', 0: 'Flutter Developer', 7: 'JavaScript Developer', 14: 'Software Engineer', 13: 'Software Engineer', 9: 'Software Engineer', 19: 'DevOps Engineer', 16: 'Wordpress Developer', 18: 'Database Administrator', 3: 'iOS Developer', 5: 'Java Developer', 6: 'Full Stack Developer', 15: 'Java Developer', 11: 'Machine Learning'}
    cluster  Job Title           Central Title
0         0        143       Flutter Developer
1         1        210        Django Developer
2         2        107        Machine Learning
3         3         66           iOS Developer
4         4        193    Full Stack Developer
5         5         95          Java Developer
6         6         77    Full Stack Developer
7         7        127    JavaScript Developer
8         8        116         DevOps Engineer
9         9        126       Software En

In [50]:
resume = extract_text('Dan Rich Resume.pdf')

#print(resume)
resume = cleantext(resume)
resume_embed = model.encode(resume)
predicted_job_cluster = kmeans.predict(resume_embed.reshape(1,-1))
#print(predicted_job_cluster)
predicted_job = title_lookup[predicted_job_cluster[0]]
print(f"Based on your resume, your skills most closely align with the profession of {predicted_job}")

Based on your resume, your skills most closely align with the profession of Machine Learning
